## Concept focus — Requirements, estimates, and architecture thinking

System design begins before diagrams. Strong design work starts by clarifying goals, constraints, traffic, failure tolerance, and trade-offs so that the architecture emerges from the problem rather than from a memorized template.

```text
vague request
    |
clarify goals / users / scale / constraints
    |
estimate load and critical paths
    |
choose architecture that fits the real problem
```

### How to think about it
Think of design as progressive constraint discovery. Before choosing databases, queues, or caches, ask what success means, what can fail, what scale matters, and what the system must optimize for under pressure.

### Visual references and further study
- [High Scalability](https://highscalability.com/)
- [System design interview primer search](https://www.youtube.com/results?search_query=system+design+fundamentals)
- [ByteByteGo blog](https://blog.bytebytego.com/)
- [Architecture trade-offs search](https://www.youtube.com/results?search_query=software+architecture+tradeoffs)

---

# Module 31 — Design Fundamentals

## Exercise 2: Back of the envelope

An estimate is not a prediction. It is a tool for eliminating options, and it
works because the answers you need are separated by orders of magnitude rather
than percentages.

"Does this fit in memory on one machine" has three possible answers: obviously
yes, obviously no, and uncomfortably close. Only the third requires care, and
knowing which one you are in takes ninety seconds.

| | |
|---|---|
| Time | About 50 minutes |
| You need | This notebook. The measurements run locally |
| Comes after | Exercise 1, requirements |

---

## 1. Round savagely

The single skill here is throwing away precision on purpose.

- A year is 30 million seconds. Really 31.5 million, and the difference never
  changes a decision.
- A day is 100,000 seconds. Really 86,400.
- A month is 2.5 million seconds.
- One million requests a day is about 12 per second. Ten million is 120.

Work in powers of ten and one significant figure. An answer of "about 100 QPS"
is useful. An answer of "116.4 QPS" is the same answer wearing a costume, and
the costume is dangerous because it invites the reader to trust the second digit.

In [ ]:
SECONDS_PER_DAY = 100_000        # actually 86,400
SECONDS_PER_MONTH = 2_500_000    # actually ~2.6 million
SECONDS_PER_YEAR = 30_000_000    # actually ~31.5 million

daily_requests = 10_000_000
print("average QPS:", daily_requests / SECONDS_PER_DAY)

The exact answer is 115.7. The estimate is 100. Every decision you would make
from that number is identical, and the estimate took no calculator.

---

## 2. Average is not the number you design for

Traffic is not flat. Real systems see a daily peak of two to ten times the
average, and the peak is what determines whether you fall over.

The rule of thumb: **plan for peak, and take peak as 2x average for a global
service, 5x for a regional one with a working-day pattern, and higher for
anything event-driven.**

Ticketing, payroll, and sports scores are not 5x. They are 100x for ten minutes
and near zero the rest of the time, which is a different kind of system and one
worth naming out loud when you see it.

In [ ]:
def qps(daily_requests, peak_multiplier=5):
    average = daily_requests / SECONDS_PER_DAY
    return average, average * peak_multiplier


avg, peak = qps(10_000_000)
print("average %.0f QPS, plan for %.0f QPS" % (avg, peak))

---

## 3. The chain: users to QPS to storage to cost

Every capacity estimate is the same four steps.

```
users  ->  requests   ->  bytes stored  ->  machines
       (per user)      (per request)     (per byte, per QPS)
```

Worked, for a photo sharing service:

- 10 million daily active users
- each views 20 photos and uploads 1, so 200 million reads and 10 million writes
  per day
- reads: 200M / 100k seconds = **2,000 QPS average, 10,000 peak**
- writes: 10M / 100k = **100 QPS average, 500 peak**
- a photo averages 2MB, so 10M x 2MB = **20TB written per day**
- at 20TB a day, a year is **7PB**

That last number is the one that matters, and it arrived in four multiplications.
7PB a year means object storage rather than a database, means a CDN rather than
serving from origin, and means the cost conversation happens now rather than in
month nine.

In [ ]:
def estimate(dau, reads_per_user, writes_per_user, bytes_per_write, peak=5):
    reads = dau * reads_per_user
    writes = dau * writes_per_user
    read_qps = reads / SECONDS_PER_DAY
    write_qps = writes / SECONDS_PER_DAY
    daily_bytes = writes * bytes_per_write
    return {
        "read QPS avg": read_qps,
        "read QPS peak": read_qps * peak,
        "write QPS avg": write_qps,
        "write QPS peak": write_qps * peak,
        "TB per day": daily_bytes / 1e12,
        "PB per year": daily_bytes * 365 / 1e15,
    }


for k, v in estimate(10_000_000, 20, 1, 2_000_000).items():
    print("%-16s %10.1f" % (k, v))

---

## 4. Latency numbers, measured rather than memorised

You have seen the table of latency numbers every programmer should know. Most
people memorise it and never check it, which means they are quoting a machine
from over a decade ago.

Measure your own. The absolute values will differ from any table. **The ratios
are what you reason with, and the ratios are stable.**

In [ ]:
import time


def time_op(fn, repeats=200_000):
    """Nanoseconds per operation, roughly. Not a benchmark, an order of magnitude."""
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    elapsed = time.perf_counter() - start
    return elapsed / repeats * 1e9


small = list(range(1000))
big = list(range(10_000_000))
d = {i: i for i in range(1_000_000)}

results = {
    "list index (cache warm)": time_op(lambda: small[500]),
    "list index (cache cold-ish)": time_op(lambda: big[9_000_000], 50_000),
    "dict lookup": time_op(lambda: d[500_000]),
    "function call": time_op(lambda: None),
}

for name, ns in results.items():
    print("%-28s %8.1f ns" % (name, ns))

Those are all in-process operations, and they are all tens of nanoseconds. Now
the ones that cross a boundary.

In [ ]:
import os
import tempfile

payload = os.urandom(1_000_000)   # 1 MB

with tempfile.NamedTemporaryFile(delete=False) as f:
    path = f.name

start = time.perf_counter()
with open(path, "wb") as f:
    f.write(payload)
    f.flush()
    os.fsync(f.fileno())
write_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
with open(path, "rb") as f:
    f.read()
read_ms = (time.perf_counter() - start) * 1000

os.unlink(path)

print("1MB durable write (fsync): %6.2f ms" % write_ms)
print("1MB read (likely cached):  %6.2f ms" % read_ms)

**The ratios worth carrying in your head**, and they hold on every machine:

| Operation | Order of magnitude |
|---|---|
| L1 cache reference | 1 ns |
| Main memory reference | 100 ns |
| SSD random read | 100,000 ns (0.1 ms) |
| Round trip within a datacentre | 500,000 ns (0.5 ms) |
| Round trip across a continent | 50,000,000 ns (50 ms) |

Memory is roughly 100x faster than SSD. A cross-continent round trip is roughly
100x a datacentre one. **Those two hundredfolds are where almost every design
decision in this course lives.** A cache is worth building because of the first;
a region is worth adding because of the second.

---

## 5. The estimate that is wrong on purpose

Here is a calculation with a mistake in it. Read it, decide where, and only then
run the cell.

> A chat service has 50 million users. Each sends 40 messages a day. A message
> averages 100 bytes. Therefore we store 50M x 40 x 100 = 200GB a day, which is
> 73TB a year, which fits comfortably on one machine.

In [ ]:
users = 50_000_000
messages_per_user_per_day = 40
bytes_per_message = 100

daily = users * messages_per_user_per_day * bytes_per_message
print("raw message bytes per day: %.0f GB" % (daily / 1e9))
print("raw message bytes per year: %.1f TB" % (daily * 365 / 1e12))

The arithmetic is right. The estimate is badly wrong, for three reasons, and
each one is a lesson.

**It counted the payload and nothing else.** A stored message carries an id, a
sender, a recipient or channel, a timestamp, delivery state, and indexes over
most of those. The real per-message cost is closer to 500 bytes than 100, so
multiply by 5 before anything else.

**It ignored replication.** Storing it once means losing it once. Three replicas
is the normal answer, so multiply by 3 again. You are now at 15x the original
number, or about 1.1PB a year.

**"Fits on one machine" answered the wrong question.** Even at 73TB, a single
machine holding all of it cannot serve 23,000 messages a second while remaining
available during a reboot. Capacity is not only about whether the bytes fit.

**The rule this gives you:** an estimate of stored data that does not include
metadata, indexes, and replication is out by roughly an order of magnitude, and
it is always out in the same direction.

In [ ]:
OVERHEAD = 5      # metadata and indexes
REPLICAS = 3

realistic = daily * OVERHEAD * REPLICAS
print("realistic: %.1f TB per day, %.2f PB per year"
      % (realistic / 1e12, realistic * 365 / 1e15))

---

# Your turn

### Task 1

Estimate, without a calculator and in under two minutes, then check with a cell.

A video platform: 500 million daily active users, each watching 5 videos a day,
average video 50MB delivered. Uploads: 1 in 1,000 users uploads one video a day.

Write your estimates in the comment **first**, then compute them.

In [ ]:
# ANSWER 1
# My guesses, before computing:
# read QPS at peak:      ___
# bandwidth in Gbps:     ___
# new storage per day:   ___
# storage per year:      ___

# Now compute:

### Task 2

Take your notification service requirements from exercise 1, task 2, and produce
its capacity estimate: QPS at peak, storage in a year, and whether the working
set fits in memory on one machine.

Then state, in one sentence, which single number most constrains the design.

In [ ]:
# ANSWER 2


most_constraining_number = "___"
because = "___"

### Task 3

Run the latency measurements from section 4 again, and answer from your own
numbers.

Then: a request handler does 200 dictionary lookups, 3 datacentre round trips,
and 1 cross-continent round trip. Where does the time go, and what is the only
change worth making?

In [ ]:
# ANSWER 3
my_dict_lookup_ns = "___"
my_1mb_fsync_ms = "___"

# 200 dict lookups + 3 local round trips + 1 cross-continent round trip
time_in_lookups_ms = "___"
time_in_local_rtts_ms = "___"
time_in_cross_continent_ms = "___"

the_only_change_worth_making = "___"
why_the_others_are_noise = "___"

### Task 4

Find someone else's estimate and break it.

Take this claim: *"Our analytics pipeline handles 1 billion events a day at
200 bytes each, so 200GB daily, so a 10TB disk gives us 50 days of retention."*

List every reason the real number is higher, and give a corrected estimate with
your multiplier for each. Then say what retention they actually have.

In [ ]:
# ANSWER 4
reasons_it_is_higher = {
    "___": "___",    # reason: multiplier
}

corrected_daily_tb = "___"
actual_retention_days = "___"

---

## Self-check

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 3, Little's Law."

a1, a2, a3, a4 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                  _answer("# ANSWER 3"), _answer("# ANSWER 4"))

results = [
    check("peak:      ___" not in a1 and "___" not in a1.split("Now compute")[0],
          "Task 1: you guessed before computing"),
    check(any(op in a1.split("Now compute")[-1] for op in ("*", "/")),
          "Task 1: you actually computed the check"),
    check(a2 and "___" not in a2, "Task 2: estimate and constraint named"),
    check(len(a2) > 200, "Task 2: the estimate has working, not just an answer"),
    check(a3 and "___" not in a3, "Task 3: answered from your own measurements"),
    check("cross" in a3.lower() or "continent" in a3.lower(),
          "Task 3: you identified the cross-continent hop as the cost"),
    check(a4.count(":") >= 4, "Task 4: at least two reasons with multipliers"),
    check(a4 and "___" not in a4, "Task 4: corrected retention given"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- Round savagely. One significant figure, powers of ten, and 100,000 seconds in
  a day.
- Design for peak, not average, and name the multiplier you assumed.
- The chain is always users to requests to bytes to machines, four
  multiplications.
- Measure latency on your own machine. Absolute values age; the ratios do not.
- Memory is about 100x faster than SSD; a cross-continent trip is about 100x a
  local one. Almost every design decision lives in those two hundredfolds.
- A storage estimate without metadata, indexes, and replication is out by roughly
  10x, always in the same direction.
- "Fits on one machine" answers a question about bytes, not about serving them.

## Before you move on

- [ ] You produced an estimate before reaching for a calculator.
- [ ] You ran the latency cells and wrote down your own numbers.
- [ ] You can state the memory-to-SSD and local-to-continental ratios from memory.
- [ ] You found at least three reasons someone else's estimate was too low.

**Next:** exercise 3, where throughput, latency, and concurrency turn out to be
three views of the same thing, and you watch a queue prove it.